# Cooperative Sahayak — ONE SINGLE FINAL NOTEBOOK

Voice-first multilingual cooperative assistance pipeline.

**Production flow**

UI-selected language → IndicConformer ASR → Tamil code-switch normalization → IndicTrans2 Indic→English → hybrid FAISS + BM25 retrieval → grounded Navarasa 2.0 answer → IndicTrans2 English→selected language → Piper TTS.

### Supported input languages
- Tamil (`ta`)
- Hindi (`hi`)
- Telugu (`te`)
- Kannada (`kn`)
- Malayalam (`ml`)

### Important runtime decisions
- The UI explicitly supplies the input language. **No automatic language detection is used in the production pipeline.**
- The existing FAISS and BM25 indexes are loaded. **Corpus embeddings are never regenerated.**
- BGE-small-en-v1.5 is used only to embed the incoming English query.
- The RAG is topic-agnostic; it is **not PMFBY-only**.
- If the retrieved evidence is not strong enough, the exact fallback is:
  `Sorry, I couldn't answer your question from the available information.`
- The project currently has Piper voices for Tamil, Hindi, Telugu, and Malayalam. **No Kannada Piper voice is configured**, so Kannada spoken output is unavailable until a Kannada voice is added.
- This notebook does not silently download models.

### Grounding safeguard
- Navarasa is explicitly prohibited from supplying unsupported legal/historical facts from pretrained knowledge; retrieved source text controls dates, numbers, names, and terminology.


## If a required model is ever missing

The following are the terminal-side resources used by this notebook. They are listed here for reproducibility; the notebook itself does not download them.

- BGE: `BAAI/bge-small-en-v1.5`
- Indic→English: `ai4bharat/indictrans2-indic-en-1B`
- English→Indic: `ai4bharat/indictrans2-en-indic-1B`
- AI4Bharat IndicConformer sherpa-onnx models
- Piper voice files already stored under `tts_models/`

The current machine was verified to contain working local snapshots for BGE and both IndicTrans2 models.


In [ ]:

# ============================================================
# 1. PROJECT CONFIGURATION
# ============================================================

from pathlib import Path

# Prefer the current notebook/project directory, with the user's
# known project location as a fallback.
PROJECT_ROOT = Path.cwd()

# Submission repository layout
if (PROJECT_ROOT / "pipeline").exists():
    PROJECT_ROOT = PROJECT_ROOT / "pipeline"

if not (PROJECT_ROOT / "data" / "rag_chunks.jsonl").exists():
    fallback_root = Path.cwd()
    if (fallback_root / "pipeline" / "data" / "rag_chunks.jsonl").exists():
        PROJECT_ROOT = fallback_root / "pipeline"

assert (PROJECT_ROOT / "data" / "rag_chunks.jsonl").exists(), (
    f"Could not find rag_chunks.jsonl under {PROJECT_ROOT}. "
    "Open this notebook from the Cooperative Sahayak project directory."
)

# -------------------------
# RAG artifacts
# -------------------------
RAG_FILE = PROJECT_ROOT / "data" / "rag_chunks.jsonl"
BGE_INDEX_DIR = PROJECT_ROOT / "indexes" / "embeddings_bge"

FAISS_FILE = BGE_INDEX_DIR / "faiss.index"
BM25_FILE = BGE_INDEX_DIR / "bm25.pkl"
METADATA_FILE = BGE_INDEX_DIR / "metadata.json"
EMBED_CONFIG_FILE = BGE_INDEX_DIR / "config.json"

# -------------------------
# External model locations
# -------------------------
# Model weights are intentionally NOT stored in this repository.
# Set these environment variables on the machine running the notebook.
#
# Example:
#   export COOP_SAHAYAK_BGE_MODEL_DIR="$HOME/.cache/huggingface/hub/..."
#   export COOP_SAHAYAK_INDIC_EN_MODEL_DIR="$HOME/.cache/huggingface/hub/..."
#   export COOP_SAHAYAK_EN_INDIC_MODEL_DIR="$HOME/.cache/huggingface/hub/..."
#   export COOP_SAHAYAK_ASR_DIR="/path/to/indic_stt_models"
#   export COOP_SAHAYAK_TTS_DIR="/path/to/tts_models"

BGE_MODEL_DIR = Path(
    os.environ.get("COOP_SAHAYAK_BGE_MODEL_DIR", "")
).expanduser()

INDIC_EN_MODEL_DIR = Path(
    os.environ.get("COOP_SAHAYAK_INDIC_EN_MODEL_DIR", "")
).expanduser()

EN_INDIC_MODEL_DIR = Path(
    os.environ.get("COOP_SAHAYAK_EN_INDIC_MODEL_DIR", "")
).expanduser()

# -------------------------
# ASR
# -------------------------
ASR_BASE_DIR = Path(
    os.environ.get("COOP_SAHAYAK_ASR_DIR", "")
).expanduser()
ASR_SAMPLE_RATE = 16000
ASR_FEATURE_DIM = 80
ASR_NUM_THREADS = 4
ASR_LANGS = ["ta", "hi", "te", "kn", "ml"]

# -------------------------
# TTS
# -------------------------
TTS_DIR = Path(
    os.environ.get("COOP_SAHAYAK_TTS_DIR", "")
).expanduser()

# Generated audio belongs outside Git history.
TTS_OUTPUT_DIR = PROJECT_ROOT / "outputs"
TTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------
# Local Navarasa llama-server
# -------------------------
LLAMA_BASE_URL = "http://127.0.0.1:8080"
LLAMA_COMPLETION_URL = f"{LLAMA_BASE_URL}/completion"

# -------------------------
# Languages
# -------------------------
SUPPORTED_LANGS = ["ta", "hi", "te", "kn", "ml"]

LANG_NAMES = {
    "ta": "Tamil",
    "hi": "Hindi",
    "te": "Telugu",
    "kn": "Kannada",
    "ml": "Malayalam",
}

TRANSLATION_SRC_LANGS = {
    "ta": "tam_Taml",
    "hi": "hin_Deva",
    "te": "tel_Telu",
    "kn": "kan_Knda",
    "ml": "mal_Mlym",
}

TRANSLATION_TARGET_LANG = "eng_Latn"
EN_INDIC_SOURCE_LANG = "eng_Latn"

EN_INDIC_TARGET_LANGS = {
    "ta": "tam_Taml",
    "hi": "hin_Deva",
    "te": "tel_Telu",
    "kn": "kan_Knda",
    "ml": "mal_Mlym",
}

# -------------------------
# Retrieval
# -------------------------
TOP_K = 3
CANDIDATE_K = 15

# Conservative evidence gate for normalized BGE cosine similarity.
# This is a retrieval safety gate, not a PMFBY/topic classifier.
RAG_MIN_DENSE_SCORE = 0.45

RAG_FALLBACK_ANSWER = (
    "Sorry, I couldn't answer your question from the available information."
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAG_FILE:", RAG_FILE)
print("BGE_MODEL_DIR:", BGE_MODEL_DIR)
print("INDIC_EN_MODEL_DIR:", INDIC_EN_MODEL_DIR)
print("EN_INDIC_MODEL_DIR:", EN_INDIC_MODEL_DIR)


In [ ]:

# ============================================================
# 2. IMPORTS AND ENVIRONMENT CHECK
# ============================================================

import os
import sys
import json
import re
import pickle
import platform
import wave

import numpy as np
import torch
import faiss
import requests

from IPython.display import Audio, display

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

from IndicTransToolkit.processor import IndicProcessor

import sherpa_onnx
import sounddevice as sd
import soundfile as sf

from piper import PiperVoice

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("FAISS:", faiss.__version__)

# Do not hard-fail on a different environment path if the notebook
# was copied elsewhere; verify the expected project environment instead.
else:
    print("⚠ Expected project Python path not found; continuing with current kernel.")

print("✓ Core imports successful")


In [ ]:

# ============================================================
# 3. VERIFY ALL LOCAL ARTIFACTS BEFORE LOADING
# ============================================================

required_paths = {
    "RAG corpus": RAG_FILE,
    "FAISS index": FAISS_FILE,
    "BM25 index": BM25_FILE,
    "BGE model": BGE_MODEL_DIR,
    "Indic→English model": INDIC_EN_MODEL_DIR,
    "English→Indic model": EN_INDIC_MODEL_DIR,
    "ASR directory": ASR_BASE_DIR,
    "TTS directory": TTS_DIR,
}

for name, path in required_paths.items():
    assert path.exists(), f"Missing {name}: {path}"
    print(f"✓ {name}: {path}")

# Verify the actual BGE weights.
assert (BGE_MODEL_DIR / "model.safetensors").exists(), (
    f"Missing BGE model weights: {BGE_MODEL_DIR / 'model.safetensors'}"
)

# Verify IndicTrans2 weights.
assert (INDIC_EN_MODEL_DIR / "model.safetensors").exists(), (
    f"Missing Indic→English weights: {INDIC_EN_MODEL_DIR / 'model.safetensors'}"
)

assert (
    (EN_INDIC_MODEL_DIR / "model.safetensors").exists()
    or (EN_INDIC_MODEL_DIR / "pytorch_model.bin").exists()
), "Missing English→Indic model weights."

print("\n✓ All required local model/index directories verified")


In [ ]:

# ============================================================
# 4. LOAD AND NORMALIZE THE RAG CORPUS
# ============================================================

records = []

with RAG_FILE.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue

        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Invalid JSON in {RAG_FILE} at line {line_number}: {e}"
            )

assert records, "RAG corpus is empty."

normalized_records = []

for record in records:
    metadata = record.get("metadata", {})

    normalized_records.append({
        "id": record.get("id", ""),
        "text": record.get("text", ""),
        "source_file": metadata.get("source_file", ""),
        "source_path": metadata.get("source_path", ""),
        "document_id": metadata.get("document_id", ""),
        "document_type": metadata.get("document_type", ""),
        "region": metadata.get("region", ""),
        "language": metadata.get("language", ""),
        "chunk_index": metadata.get("chunk_index", ""),
        "total_chunks": metadata.get("total_chunks", ""),
        "content_hash": metadata.get("content_hash", ""),
    })

assert len(normalized_records) == len(records)
assert all(r["text"].strip() for r in normalized_records), (
    "At least one corpus record has empty text."
)

print("✓ Corpus records:", len(normalized_records))
print("✓ Corpus normalized")


In [ ]:

# ============================================================
# 5. LOAD EXISTING FAISS + BM25 + BGE QUERY ENCODER
# ============================================================
# IMPORTANT:
# - FAISS is loaded from embeddings_bge/faiss.index.
# - BM25 is loaded from embeddings_bge/bm25.pkl.
# - Corpus embeddings are NEVER regenerated.
# - BGE is used only for NEW QUERY embeddings.

faiss_index = faiss.read_index(str(FAISS_FILE))

assert faiss_index.ntotal == len(normalized_records), (
    f"FAISS vectors ({faiss_index.ntotal}) != "
    f"corpus chunks ({len(normalized_records)})"
)
assert faiss_index.d == 384, (
    f"Expected BGE dimension 384, got {faiss_index.d}"
)

with BM25_FILE.open("rb") as f:
    bm25 = pickle.load(f)

assert len(bm25.doc_freqs) == len(normalized_records), (
    f"BM25 documents ({len(bm25.doc_freqs)}) != "
    f"corpus chunks ({len(normalized_records)})"
)

embed_model = SentenceTransformer(
    str(BGE_MODEL_DIR),
    device="cpu",
)

assert embed_model.get_embedding_dimension() == faiss_index.d

# Query-only smoke test.
test_embedding = embed_model.encode(
    ["test query"],
    normalize_embeddings=True,
    convert_to_numpy=True,
)

assert test_embedding.shape == (1, faiss_index.d)

print("✓ Existing FAISS loaded")
print("✓ Existing BM25 loaded")
print("✓ Local BGE loaded")
print("✓ BGE query embedding dimension:", test_embedding.shape[1])
print("✓ Corpus embeddings were NOT regenerated")


In [ ]:

# ============================================================
# 6. HYBRID RETRIEVAL
# ============================================================

BM25_STOPWORDS = {
    "what", "is", "the", "a", "an", "are", "was", "were",
    "be", "been", "being", "of", "to", "in", "on", "for",
    "from", "by", "with", "and", "or", "as", "at", "this",
    "that", "these", "those", "it", "its", "how", "why",
    "when", "where", "which", "who", "does", "do", "did",
    "can", "could", "should", "would", "will",
}

def bm25_tokenize(text):
    tokens = re.findall(r"[a-z0-9]+", str(text).lower())
    return [t for t in tokens if t not in BM25_STOPWORDS]


def hybrid_search(query, top_k=TOP_K, candidate_k=CANDIDATE_K):
    if not isinstance(query, str) or not query.strip():
        return []

    # BM25 candidate retrieval.
    bm25_scores = bm25.get_scores(bm25_tokenize(query))
    bm25_indices = np.argsort(bm25_scores)[::-1][:candidate_k]

    # BGE query embedding only.
    query_embedding = embed_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    dense_scores, dense_indices = faiss_index.search(
        query_embedding,
        candidate_k,
    )

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    candidate_indices = {
        int(i) for i in bm25_indices
    }
    candidate_indices.update(
        int(i) for i in dense_indices if i >= 0
    )

    bm25_rank_map = {
        int(i): rank
        for rank, i in enumerate(bm25_indices, start=1)
    }

    dense_score_map = {
        int(i): float(score)
        for i, score in zip(dense_indices, dense_scores)
        if i >= 0
    }

    # Preserve the project's semantic-first hybrid behavior:
    # BGE score is primary; BM25 rank breaks ties / expands candidates.
    ranked = sorted(
        candidate_indices,
        key=lambda i: (
            dense_score_map.get(i, -1.0),
            -bm25_rank_map.get(i, 999999),
        ),
        reverse=True,
    )[:top_k]

    results = []

    for rank, index in enumerate(ranked, start=1):
        record = normalized_records[index]

        results.append({
            "rank": rank,
            "index": index,
            "id": record["id"],
            "text": record["text"],
            "source_file": record["source_file"],
            "source_path": record["source_path"],
            "document_id": record["document_id"],
            "document_type": record["document_type"],
            "region": record["region"],
            "language": record["language"],
            "chunk_index": record["chunk_index"],
            "total_chunks": record["total_chunks"],
            "dense_score": dense_score_map.get(index, 0.0),
            "bm25_rank": bm25_rank_map.get(index),
        })

    return results


def format_source_attribution(results):
    if not results:
        return "No sources retrieved."

    lines = []
    for r in results:
        lines.append(
            f"- {r['source_file']} | "
            f"Chunk {r['chunk_index']}/{r['total_chunks']} | "
            f"BGE score {r['dense_score']:.4f}"
        )
    return "\n".join(lines)


print("✓ Hybrid FAISS + BM25 retrieval ready")


In [ ]:

# ============================================================
# 7. INDIC → ENGLISH TRANSLATION
# ============================================================

TRANSLATION_DEVICE = "cpu"

translator_tokenizer = AutoTokenizer.from_pretrained(
    str(INDIC_EN_MODEL_DIR),
    local_files_only=False,
    trust_remote_code=True,
)

translator_model = AutoModelForSeq2SeqLM.from_pretrained(
    str(INDIC_EN_MODEL_DIR),
    local_files_only=False,
    trust_remote_code=True,
)

translator_model = translator_model.to(TRANSLATION_DEVICE)
translator_model.eval()

indic_processor = IndicProcessor(inference=True)


def translate_to_english(text, language_code, max_length=128):
    if language_code not in TRANSLATION_SRC_LANGS:
        raise ValueError(
            f"Unsupported language: {language_code}"
        )

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text must be a non-empty string.")

    src_lang = TRANSLATION_SRC_LANGS[language_code]

    batch = indic_processor.preprocess_batch(
        [text],
        src_lang=src_lang,
        tgt_lang=TRANSLATION_TARGET_LANG,
    )

    inputs = translator_tokenizer(
        batch,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(TRANSLATION_DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        generated_tokens = translator_model.generate(
            **inputs,
            use_cache=False,
            max_length=max_length,
            num_beams=1,
            do_sample=False,
            num_return_sequences=1,
        )

    decoded = translator_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    translated = indic_processor.postprocess_batch(
        decoded,
        lang=TRANSLATION_TARGET_LANG,
    )

    return translated[0].strip()


print("✓ Indic→English model loaded locally")
print("✓ Indic→English translation function ready")


In [ ]:

# ============================================================
# 8. ENGLISH → INDIC TRANSLATION
# ============================================================

en_indic_tokenizer = AutoTokenizer.from_pretrained(
    str(EN_INDIC_MODEL_DIR),
    local_files_only=False,
    trust_remote_code=True,
)

en_indic_model = AutoModelForSeq2SeqLM.from_pretrained(
    str(EN_INDIC_MODEL_DIR),
    local_files_only=False,
    trust_remote_code=True,
)

en_indic_model = en_indic_model.to(TRANSLATION_DEVICE)
en_indic_model.eval()

en_indic_processor = IndicProcessor(inference=True)


def translate_english_to_indic(text, language_code, max_length=512):
    if language_code not in EN_INDIC_TARGET_LANGS:
        raise ValueError(
            f"Unsupported language: {language_code}"
        )

    if not isinstance(text, str) or not text.strip():
        raise ValueError("Input text must be a non-empty string.")

    tgt_lang = EN_INDIC_TARGET_LANGS[language_code]

    batch = en_indic_processor.preprocess_batch(
        [text],
        src_lang=EN_INDIC_SOURCE_LANG,
        tgt_lang=tgt_lang,
    )

    inputs = en_indic_tokenizer(
        batch,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(TRANSLATION_DEVICE)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        generated_tokens = en_indic_model.generate(
            **inputs,
            use_cache=False,
            max_length=max_length,
            num_beams=1,
            do_sample=False,
            num_return_sequences=1,
        )

    decoded = en_indic_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    translated = en_indic_processor.postprocess_batch(
        decoded,
        lang=tgt_lang,
    )

    return translated[0].strip()


print("✓ English→Indic model loaded locally")
print("✓ English→Indic translation function ready")


In [ ]:

# ============================================================
# 9. LOCAL NAVARASA 2.0 SERVER
# ============================================================

def check_navarasa_server():
    try:
        response = requests.get(
            LLAMA_BASE_URL,
            timeout=3,
        )
        return True, response.status_code
    except Exception as e:
        return False, str(e)


server_ok, server_info = check_navarasa_server()

if server_ok:
    print(f"✓ Navarasa server reachable: HTTP {server_info}")
else:
    print(
        "⚠ Navarasa server is not currently reachable.\n"
        "Start your existing llama-server on 127.0.0.1:8080 "
        "before running the full voice pipeline."
    )


def generate_with_navarasa(
    prompt,
    n_predict=320,
    temperature=0.0,
    timeout=120,
):
    response = requests.post(
        LLAMA_COMPLETION_URL,
        json={
            "prompt": prompt,
            "n_predict": n_predict,
            "temperature": temperature,
            "top_p": 0.8,
            "repeat_penalty": 1.20,
            "stream": False,
            "stop": [
                "</s>",
                "RETRIEVED SOURCE TEXT:",
                "USER QUESTION:",
                "ANSWER:",
            ],
        },
        timeout=timeout,
    )

    response.raise_for_status()

    answer = response.json().get("content", "").strip()
    answer = answer.strip("= \n\t")

    if answer.lower().startswith("answer:"):
        answer = answer[7:].strip()

    return answer


In [ ]:
# ============================================================
# CELL 12 — GENERIC GROUNDED RAG + NAVARASA
# ============================================================

RAG_FALLBACK = "Sorry, I couldn't answer your question from the available information."


def build_grounded_context(results):
    """
    Build context with the highest-ranked retrieved chunk clearly identified
    as the PRIMARY SOURCE.

    The existing retrieval system is unchanged:
    FAISS/BGE + BM25 still determines the ranking.
    """

    if not results:
        return ""

    primary = results[0]

    def get_text(item):
        return str(
            item.get("text")
            or item.get("chunk_text")
            or item.get("content")
            or ""
        ).strip()

    def get_source(item):
        return str(
            item.get("source")
            or item.get("source_file")
            or item.get("metadata", {}).get("source_file", "")
            or ""
        )

    def get_chunk(item):
        return str(
            item.get("chunk_id")
            or item.get("chunk_index")
            or item.get("metadata", {}).get("chunk_index", "")
        )

    primary_text = get_text(primary)

    if not primary_text:
        return ""

    context = (
        "PRIMARY SOURCE\n"
        f"Source: {get_source(primary)}\n"
        f"Chunk: {get_chunk(primary)}\n\n"
        f"{primary_text}"
    )

    # Supporting chunks are retained, but explicitly separated from the
    # primary source so the model does not casually mix unrelated historical
    # numbers/dates into the main answer.
    supporting = []

    for item in results[1:]:
        text = get_text(item)
        if not text:
            continue

        supporting.append(
            "SUPPORTING SOURCE\n"
            f"Source: {get_source(item)}\n"
            f"Chunk: {get_chunk(item)}\n\n"
            f"{text}"
        )

    if supporting:
        context += "\n\n" + "\n\n".join(supporting)

    return context


def navarasa_completion(prompt, max_tokens=320, temperature=0.0):
    """Call the local Navarasa llama-server."""

    payload = {
        "prompt": prompt,
        "n_predict": max_tokens,
        "temperature": temperature,
        "top_p": 0.9,
        "stop": [
            "\n\nUSER QUESTION:",
            "\n\nANSWER:",
            "\n\nFINAL ANSWER:",
            "<|eot_id|>",
            "</s>",
        ],
    }

    response = requests.post(
        LLAMA_COMPLETION_URL,
        json=payload,
        timeout=120,
    )
    response.raise_for_status()

    data = response.json()

    answer = data.get("content", "")
    if not answer:
        answer = data.get("response", "")

    return str(answer).strip()


def clean_navarasa_answer(answer):
    """Remove accidental answer labels without changing the answer itself."""

    if not answer:
        return ""

    answer = answer.strip()

    for prefix in ("ANSWER:", "Answer:", "answer:", "FINAL ANSWER:"):
        if answer.startswith(prefix):
            answer = answer[len(prefix):].strip()

    return answer


def is_complete_answer(answer):
    """
    Reject obvious truncation only.

    A short but valid one-sentence answer is accepted.
    """

    if not answer:
        return False

    if answer == RAG_FALLBACK:
        return True

    text = answer.strip()

    # Obvious mid-sentence endings.
    if text.endswith((",", ":", ";", "-", "—", "–", "/", "(")):
        return False

    # Unbalanced brackets usually indicate generation truncation.
    for opening, closing in (
        ("(", ")"),
        ("[", "]"),
        ("{", "}"),
    ):
        if text.count(opening) != text.count(closing):
            return False

    # Require normal sentence completion.
    if text[-1] not in ".!?":
        return False

    return True


def generate_grounded_answer(english_query, results):
    """
    Generate a generic answer grounded only in retrieved evidence.

    The primary retrieved chunk is the main evidence.
    """

    context = build_grounded_context(results)

    if not context:
        return RAG_FALLBACK

    prompt = f"""You are Cooperative Sahayak, a grounded information assistant.

Answer the user's question ONLY using the retrieved source text below.

STRICT GROUNDING RULES:
1. Every factual statement must be directly supported by the retrieved text.
2. Do NOT use pretrained knowledge or outside knowledge.
3. Do NOT guess missing information.
4. Do NOT invent dates, years, Act numbers, section numbers, names,
   institutions, amounts, procedures, or legal claims.
5. Do NOT combine a number or date from one source/chunk with a different
   subject from another source/chunk.
6. PRIMARY SOURCE is the main evidence and should be used first.
7. SUPPORTING SOURCE material may be used only when it directly supports
   the answer.
8. Do not "correct" the source using outside knowledge.
9. Answer directly and concisely in 2–3 complete sentences when the source
   supports that much information.
10. Finish every sentence completely. Never stop after a comma, date, number,
    colon, or other partial phrase.
11. If the retrieved text does not clearly answer the question, output exactly:

Sorry, I couldn't answer your question from the available information.

RETRIEVED SOURCE TEXT:

{context}

USER QUESTION:
{english_query}

ANSWER:
"""

    answer = navarasa_completion(
        prompt,
        max_tokens=320,
        temperature=0.0,
    )
    answer = clean_navarasa_answer(answer)

    # One controlled retry for obvious truncation.
    if not is_complete_answer(answer):
        retry_prompt = f"""You are Cooperative Sahayak.

Produce a COMPLETE answer to the user's question using ONLY the retrieved
source text.

RULES:
- Use no outside or pretrained knowledge.
- Do not invent or infer facts.
- PRIMARY SOURCE has priority.
- Do not mix unrelated dates, years, Act numbers, or section numbers.
- Give 2–3 complete sentences if supported by the source.
- Finish the answer completely.
- Never end in the middle of a sentence.
- If the source does not clearly answer the question, output exactly:

Sorry, I couldn't answer your question from the available information.

RETRIEVED SOURCE TEXT:

{context}

USER QUESTION:
{english_query}

FINAL ANSWER:
"""

        retry = navarasa_completion(
            retry_prompt,
            max_tokens=320,
            temperature=0.0,
        )
        retry = clean_navarasa_answer(retry)

        if is_complete_answer(retry):
            answer = retry
        else:
            answer = RAG_FALLBACK

    return answer


def run_grounded_rag(english_query, results):
    print("\nSTEP 5 — Grounded answer")

    answer = generate_grounded_answer(
        english_query,
        results,
    )

    print("English answer:")
    print(answer)

    return answer


In [ ]:

# ============================================================
# 11. AI4BHARAT INDICCONFORMER ASR
# ============================================================

ASR_LANG_NAMES = {
    "ta": "Tamil",
    "hi": "Hindi",
    "te": "Telugu",
    "kn": "Kannada",
    "ml": "Malayalam",
}

ASR_RECOGNIZERS = {}
ASR_TOKENS_PATH = ASR_BASE_DIR / "tokens.txt"

assert ASR_TOKENS_PATH.exists(), f"Missing ASR tokens: {ASR_TOKENS_PATH}"

for lang in ASR_LANGS:
    model_path = ASR_BASE_DIR / lang / "model.int8.onnx"

    assert model_path.exists(), (
        f"Missing IndicConformer model: {model_path}"
    )

    ASR_RECOGNIZERS[lang] = sherpa_onnx.OfflineRecognizer.from_nemo_ctc(
        model=str(model_path),
        tokens=str(ASR_TOKENS_PATH),
        num_threads=ASR_NUM_THREADS,
        sample_rate=ASR_SAMPLE_RATE,
        feature_dim=ASR_FEATURE_DIM,
        debug=False,
        provider="cpu",
    )

    print(f"✓ {ASR_LANG_NAMES[lang]} ASR initialized")

assert set(ASR_RECOGNIZERS) == set(SUPPORTED_LANGS)


def transcribe_audio(
    audio_data,
    sample_rate=ASR_SAMPLE_RATE,
    lang="ta",
):
    if lang not in ASR_RECOGNIZERS:
        raise ValueError(
            f"Unsupported ASR language: {lang}"
        )

    audio_data = np.asarray(
        audio_data,
        dtype=np.float32,
    ).flatten()

    if audio_data.size == 0:
        return ""

    recognizer = ASR_RECOGNIZERS[lang]
    stream = recognizer.create_stream()

    stream.accept_waveform(
        sample_rate,
        audio_data,
    )

    recognizer.decode_stream(stream)

    return stream.result.text.strip()


print("✓ IndicConformer ASR ready")


In [ ]:

# ============================================================
# 12. TAMIL CODE-SWITCH NORMALIZATION
# ============================================================
# Restored from the earlier project notebook rather than using
# the later identity-only placeholder.

TAMIL_CODE_SWITCH_MAP = {
    "லோவன்": "loan",
    "லோன்": "loan",
    "ஆப்பிளி": "apply",
    "அப்பிளை": "apply",
    "டோகுமெண்ட்ஸ்": "documents",
    "டாக்குமெண்ட்ஸ்": "documents",
}


def normalize_tamil_code_switch(text):
    if text is None:
        return ""

    text = str(text).strip()

    if not text:
        return ""

    normalized = text

    for tamil_word, english_word in TAMIL_CODE_SWITCH_MAP.items():
        normalized = normalized.replace(
            tamil_word,
            english_word,
        )

    return normalized


def transcribe_and_normalize(
    audio_data,
    sample_rate,
    language_code,
):
    transcription = transcribe_audio(
        audio_data=audio_data,
        sample_rate=sample_rate,
        lang=language_code,
    )

    normalized = transcription

    # The existing normalization map is specifically for
    # Tamil code-switched ASR. Other languages pass through.
    if language_code == "ta":
        normalized = normalize_tamil_code_switch(transcription)

    return transcription, normalized


print("✓ Tamil code-switch normalization ready")


In [ ]:

# ============================================================
# 13. PIPER TTS CONFIGURATION
# ============================================================

TTS_MODEL_FILES = {
    "ta": TTS_DIR / "ta_IN-rasa_female-medium.onnx",
    "hi": (
        TTS_DIR / "hi" / "hi_IN" / "priyamvada" / "medium"
        / "hi_IN-priyamvada-medium.onnx"
    ),
    "te": (
        TTS_DIR / "te" / "te_IN" / "venkatesh" / "medium"
        / "te_IN-venkatesh-medium.onnx"
    ),
    "ml": (
        TTS_DIR / "ml" / "ml_IN" / "meera" / "medium"
        / "ml_IN-meera-medium.onnx"
    ),
}

TTS_LANGUAGE_NAMES = {
    "ta": "Tamil",
    "hi": "Hindi",
    "te": "Telugu",
    "kn": "Kannada",
    "ml": "Malayalam",
}

TTS_VOICES = {}

for lang, model_path in TTS_MODEL_FILES.items():
    config_path = Path(str(model_path) + ".json")

    assert model_path.exists(), (
        f"Missing TTS model: {model_path}"
    )
    assert config_path.exists(), (
        f"Missing TTS config: {config_path}"
    )

    TTS_VOICES[lang] = PiperVoice.load(str(model_path))
    print(f"✓ {TTS_LANGUAGE_NAMES[lang]} Piper voice loaded")

print(
    "\n⚠ Kannada has no configured Piper voice in this project. "
    "Kannada input/translation is supported, but spoken Kannada output "
    "cannot be generated until a Kannada Piper model is supplied."
)


In [ ]:

# ============================================================
# 14. TTS FUNCTIONS
# ============================================================

def validate_tts_language(language_code):
    if language_code not in TTS_VOICES:
        available = ", ".join(TTS_VOICES.keys())
        raise ValueError(
            f"No Piper voice is configured for '{language_code}'. "
            f"Available spoken-output languages: {available}"
        )
    return True


def prepare_tts_text(text):
    if text is None:
        raise ValueError("TTS text is empty.")

    text = " ".join(str(text).strip().split())

    if not text:
        raise ValueError("TTS text is empty.")

    text = text.replace(".", ". ")
    text = text.replace("?", "? ")
    text = text.replace("!", "! ")

    return " ".join(text.split())


def synthesize_tts(
    text,
    language_code,
    output_filename=None,
):
    validate_tts_language(language_code)

    if not text or not text.strip():
        raise ValueError("TTS text is empty.")

    if output_filename is None:
        output_filename = (
            TTS_OUTPUT_DIR / f"tts_{language_code}.wav"
        )
    else:
        output_filename = Path(output_filename)

    output_filename.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with wave.open(str(output_filename), "wb") as wav_file:
        TTS_VOICES[language_code].synthesize_wav(
            text.strip(),
            wav_file,
        )

    return output_filename


def generate_final_spoken_answer(
    english_answer,
    language_code,
):
    if not english_answer or not english_answer.strip():
        raise ValueError("English answer is empty.")

    validate_tts_language(language_code)

    indic_text = translate_english_to_indic(
        english_answer,
        language_code,
        max_length=512,
    )

    paced_text = prepare_tts_text(indic_text)

    audio_file = synthesize_tts(
        paced_text,
        language_code,
        TTS_OUTPUT_DIR / f"final_spoken_answer_{language_code}.wav",
    )

    return indic_text, audio_file


print("✓ Piper TTS functions ready")


In [ ]:

# ============================================================
# 15. FINAL END-TO-END PIPELINE
# ============================================================
# IMPORTANT:
# The language_code comes from the UI.
# No automatic language identification is called here.

def run_full_pipeline(
    audio_data,
    language_code,
    sample_rate=ASR_SAMPLE_RATE,
):
    if audio_data is None or len(audio_data) == 0:
        raise ValueError("Input audio is empty.")

    if language_code not in SUPPORTED_LANGS:
        raise ValueError(
            f"Unsupported language '{language_code}'. "
            f"Choose from {SUPPORTED_LANGS}."
        )

    print("=" * 60)
    print("COOPERATIVE SAHAYAK")
    print("=" * 60)

    print("\nSTEP 1 — UI-selected input language")
    print(f"✓ {LANG_NAMES[language_code]} ({language_code})")

    print("\nSTEP 2 — Speech to text")
    native_text, normalized_text = transcribe_and_normalize(
        audio_data=audio_data,
        sample_rate=sample_rate,
        language_code=language_code,
    )

    print("Raw ASR:")
    print(native_text or "[No speech detected]")

    print("\nNormalized ASR:")
    print(normalized_text or "[No speech detected]")

    if not normalized_text.strip():
        raise ValueError("Speech could not be transcribed.")

    print("\nSTEP 3 — Indic → English")
    english_query = translate_to_english(
        normalized_text,
        language_code,
    )

    print("English query:")
    print(english_query)

    if not english_query.strip():
        raise ValueError("English translation is empty.")

    print("\nSTEP 4 — Hybrid knowledge retrieval")
    retrieved_results = hybrid_search(
        english_query,
        top_k=TOP_K,
        candidate_k=CANDIDATE_K,
    )

    print(f"Retrieved: {len(retrieved_results)} chunks")

    if retrieved_results:
        print(f"Top BGE score: {retrieved_results[0]['dense_score']:.4f}")
        print("\nSources:")
        print(format_source_attribution(retrieved_results))

    print("\nSTEP 5 — Grounded answer")
    english_answer = generate_grounded_answer(
        english_query,
        retrieved_results,
    )

    print("English answer:")
    print(english_answer)

    # Text answer is always available if translation is available.
    print("\nSTEP 6 — English → selected Indic language")

    indic_answer = translate_english_to_indic(
        english_answer,
        language_code,
        max_length=512,
    )

    print("Indic answer:")
    print(indic_answer)

    # Spoken output is only generated where this project has a Piper voice.
    audio_file = None

    if language_code in TTS_VOICES:
        print("\nSTEP 7 — Piper TTS")
        paced_text = prepare_tts_text(indic_answer)

        audio_file = synthesize_tts(
            paced_text,
            language_code,
            TTS_OUTPUT_DIR / f"final_spoken_answer_{language_code}.wav",
        )

        print("Audio:", audio_file)

        print("\nSTEP 8 — Play output")
        display(Audio(filename=str(audio_file)))
    else:
        print(
            "\nSTEP 7 — Piper TTS skipped"
            "\n⚠ No Kannada Piper voice is configured."
        )

    print("\n" + "=" * 60)
    print("✓ PIPELINE COMPLETED")
    print("=" * 60)

    return {
        "input_language": language_code,
        "native_transcription": native_text,
        "normalized_transcription": normalized_text,
        "english_query": english_query,
        "retrieved_results": retrieved_results,
        "english_answer": english_answer,
        "indic_answer": indic_answer,
        "audio_file": audio_file,
    }


print("✓ Final production pipeline ready")


In [ ]:

# ============================================================
# 16. FINAL PRE-FLIGHT
# ============================================================

assert set(SUPPORTED_LANGS) == {
    "ta", "hi", "te", "kn", "ml"
}

assert set(ASR_RECOGNIZERS) == set(SUPPORTED_LANGS)
assert set(TRANSLATION_SRC_LANGS) == set(SUPPORTED_LANGS)
assert set(EN_INDIC_TARGET_LANGS) == set(SUPPORTED_LANGS)

assert faiss_index.ntotal == len(normalized_records)
assert faiss_index.d == 384
assert embed_model.get_embedding_dimension() == 384

assert translator_model is not None
assert translator_tokenizer is not None
assert indic_processor is not None

assert en_indic_model is not None
assert en_indic_tokenizer is not None
assert en_indic_processor is not None

assert TTS_OUTPUT_DIR.exists()
assert TTS_VOICES

print("=" * 60)
print("FINAL PRE-FLIGHT PASSED")
print("=" * 60)
print("✓ UI-selected language architecture")
print("✓ IndicConformer ASR")
print("✓ Tamil code-switch normalization")
print("✓ IndicTrans2 Indic→English")
print("✓ Existing FAISS")
print("✓ Existing BM25")
print("✓ BGE query encoder")
print("✓ Generic grounded RAG")
print("✓ Navarasa integration")
print("✓ English→Indic")
print("✓ Piper TTS where a voice exists")
print("✓ No corpus embedding regeneration")
print("✓ No production automatic language detection")


## Run the live microphone pipeline

The cell below is intentionally the **only live microphone execution cell**.

Set the language using the same value your UI button supplies:
`ta`, `hi`, `te`, `kn`, or `ml`.

For Kannada, the pipeline can transcribe, retrieve, answer, and translate to Kannada, but this project currently has no Kannada Piper voice, so it will return the Kannada text without spoken audio.


In [ ]:

# ============================================================
# 17. LIVE MICROPHONE — UI LANGUAGE SELECTION
# ============================================================

VOICE_TEST_DURATION = 7

print("=" * 60)
print("COOPERATIVE SAHAYAK — LIVE VOICE PIPELINE")
print("=" * 60)

print("\nSupported UI language codes:")
for code, name in LANG_NAMES.items():
    status = "spoken output available" if code in TTS_VOICES else "text output only"
    print(f"  {code} -> {name} ({status})")

SELECTED_LANGUAGE = input(
    "\nEnter the language selected by the UI "
    "(ta/hi/te/kn/ml): "
).strip().lower()

if SELECTED_LANGUAGE not in SUPPORTED_LANGS:
    raise ValueError(
        f"Unsupported language '{SELECTED_LANGUAGE}'. "
        f"Choose from {SUPPORTED_LANGS}."
    )

print(f"\n✓ UI-selected language: {LANG_NAMES[SELECTED_LANGUAGE]}")

input("\nPress ENTER and then speak your question...")

print(f"\nRecording for {VOICE_TEST_DURATION} seconds...")

audio = sd.rec(
    int(VOICE_TEST_DURATION * ASR_SAMPLE_RATE),
    samplerate=ASR_SAMPLE_RATE,
    channels=1,
    dtype="float32",
)

sd.wait()

audio = audio.flatten()

print("Recording finished.")

result = run_full_pipeline(
    audio_data=audio,
    language_code=SELECTED_LANGUAGE,
    sample_rate=ASR_SAMPLE_RATE,
)

print("\nFINAL RESULT")
print("Language:", result["input_language"])
print("Native transcription:", result["native_transcription"])
print("Normalized transcription:", result["normalized_transcription"])
print("English query:", result["english_query"])
print("English answer:", result["english_answer"])
print("Indic answer:", result["indic_answer"])
print("Audio:", result["audio_file"])


In [ ]:
%pip install pandas numpy jiwer scikit-learn

In [ ]:
# ============================================================
# COOPERATIVE SAHAYAK — COMPLETE METRICS / EVALUATION CELL
# ============================================================
#
# Run this AFTER Cell 19 has successfully completed.
#
# It measures:
#   1. ASR WER / CER / accuracy-like score
#   2. ASR latency
#   3. Indic -> English translation latency
#   4. RAG retrieval latency
#   5. RAG BGE score
#   6. RAG BM25 contribution / rank
#   7. Navarasa LLM latency
#   8. Navarasa input/output token usage (when server reports it)
#   9. English -> Indic translation latency
#  10. Piper TTS latency
#  11. End-to-end latency
#
# Output:
#   - Per-run metrics table
#   - Summary statistics
#   - cooperative_sahayak_metrics.csv
# ============================================================

import time
import json
import numpy as np
import pandas as pd
from pathlib import Path
from jiwer import wer, cer


# ============================================================
# SETTINGS
# ============================================================

NUM_RUNS = 5

METRICS_FILE = PROJECT_ROOT / "evaluation" / "cooperative_sahayak_metrics.csv"

print("=" * 75)
print("COOPERATIVE SAHAYAK — SYSTEM PERFORMANCE EVALUATION")
print("=" * 75)

print(f"\nLanguage: {LANG_NAMES.get(SELECTED_LANGUAGE, SELECTED_LANGUAGE)}")
print(f"Number of evaluation runs: {NUM_RUNS}")

if "audio" not in globals():
    raise RuntimeError(
        "No 'audio' variable found. Run Cell 19 first and record an audio sample."
    )

if "SELECTED_LANGUAGE" not in globals():
    raise RuntimeError(
        "SELECTED_LANGUAGE not found. Run Cell 19 first."
    )


# ============================================================
# ASR GROUND TRUTH
# ============================================================
#
# Enter the HUMAN-CORRECT transcription of the audio recorded
# in Cell 19.
#
# Example:
# Tamil speech: "எனது கடன் இருப்பு என்ன"
#
# Enter exactly what was spoken.
# ============================================================

REFERENCE_TEXT = input(
    "\nEnter the correct HUMAN transcription of the recorded audio "
    "(press ENTER to skip ASR accuracy):\n> "
).strip()

if REFERENCE_TEXT:
    print("\n✓ Ground-truth transcription received.")
else:
    print(
        "\n⚠ No ground truth supplied."
        "\nWER/CER/ASR accuracy will be shown as NaN."
        "\nLatency metrics will still be measured."
    )


# ============================================================
# STORAGE
# ============================================================

records = []


# ============================================================
# HELPER — EXTRACT TOKEN COUNTS FROM LLAMA-SERVER RESPONSE
# ============================================================

def extract_llm_tokens(data):

    input_tokens = np.nan
    output_tokens = np.nan

    # Common llama.cpp / llama-server naming
    possible_input_keys = [
        "tokens_evaluated",
        "prompt_tokens",
        "input_tokens",
    ]

    possible_output_keys = [
        "tokens_predicted",
        "completion_tokens",
        "output_tokens",
    ]

    for key in possible_input_keys:
        if key in data:
            try:
                input_tokens = int(data[key])
                break
            except Exception:
                pass

    for key in possible_output_keys:
        if key in data:
            try:
                output_tokens = int(data[key])
                break
            except Exception:
                pass

    # OpenAI-style usage object
    usage = data.get("usage", {})

    if isinstance(usage, dict):

        if pd.isna(input_tokens):
            for key in ["prompt_tokens", "input_tokens"]:
                if key in usage:
                    try:
                        input_tokens = int(usage[key])
                        break
                    except Exception:
                        pass

        if pd.isna(output_tokens):
            for key in ["completion_tokens", "output_tokens"]:
                if key in usage:
                    try:
                        output_tokens = int(usage[key])
                        break
                    except Exception:
                        pass

    total_tokens = np.nan

    if not pd.isna(input_tokens) and not pd.isna(output_tokens):
        total_tokens = input_tokens + output_tokens

    return input_tokens, output_tokens, total_tokens


# ============================================================
# HELPER — INSTRUMENT NAVARASA
# ============================================================
#
# Your production function returns only text.
# This temporary wrapper captures token usage and latency while
# keeping the actual prompt/configuration unchanged.
# ============================================================

original_navarasa_completion = navarasa_completion


def measured_navarasa_completion(
    prompt,
    max_tokens=320,
    temperature=0.0,
):

    start = time.perf_counter()

    response = requests.post(
        LLAMA_COMPLETION_URL,
        json={
            "prompt": prompt,
            "n_predict": max_tokens,
            "temperature": temperature,
            "top_p": 0.9,
            "stop": [
                "\n\nUSER QUESTION:",
                "\n\nANSWER:",
                "\n\nFINAL ANSWER:",
                "<|eot_id|>",
                "</s>",
            ],
        },
        timeout=120,
    )

    response.raise_for_status()

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1000

    data = response.json()

    answer = data.get("content", "")

    if not answer:
        answer = data.get("response", "")

    input_tokens, output_tokens, total_tokens = (
        extract_llm_tokens(data)
    )

    # Store information for the current evaluation run.
    measured_navarasa_completion.last_metrics = {
        "latency_ms": elapsed_ms,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
    }

    return str(answer).strip()


# ============================================================
# EVALUATION LOOP
# ============================================================

print("\n")
print("=" * 75)
print("STARTING EVALUATION")
print("=" * 75)


for run_number in range(1, NUM_RUNS + 1):

    print(f"\n{'─' * 75}")
    print(f"RUN {run_number}/{NUM_RUNS}")
    print(f"{'─' * 75}")

    metrics = {
        "run": run_number,
        "language": SELECTED_LANGUAGE,
    }


    # --------------------------------------------------------
    # TOTAL PIPELINE TIMER
    # --------------------------------------------------------

    total_start = time.perf_counter()


    # --------------------------------------------------------
    # STEP 1 — ASR
    # --------------------------------------------------------

    asr_start = time.perf_counter()

    native_text = transcribe_audio(
        audio_data=audio,
        sample_rate=ASR_SAMPLE_RATE,
        lang=SELECTED_LANGUAGE,
    )

    asr_latency_ms = (
        time.perf_counter() - asr_start
    ) * 1000

    metrics["asr_latency_ms"] = asr_latency_ms
    metrics["asr_text"] = native_text

    # ASR quality
    if REFERENCE_TEXT:

        try:
            metrics["wer"] = wer(
                REFERENCE_TEXT,
                native_text,
            )

            metrics["cer"] = cer(
                REFERENCE_TEXT,
                native_text,
            )

            metrics["asr_accuracy_percent"] = (
                max(0, 1 - metrics["wer"])
                * 100
            )

        except Exception:
            metrics["wer"] = np.nan
            metrics["cer"] = np.nan
            metrics["asr_accuracy_percent"] = np.nan

    else:

        metrics["wer"] = np.nan
        metrics["cer"] = np.nan
        metrics["asr_accuracy_percent"] = np.nan


    # --------------------------------------------------------
    # CODE-SWITCH NORMALIZATION
    # --------------------------------------------------------

    normalized_text = native_text

    if SELECTED_LANGUAGE == "ta":

        normalized_text = (
            normalize_tamil_code_switch(
                native_text
            )
        )

    metrics["normalized_text"] = normalized_text


    # --------------------------------------------------------
    # STEP 2 — INDIC → ENGLISH
    # --------------------------------------------------------

    translation_start = time.perf_counter()

    english_query = translate_to_english(
        normalized_text,
        SELECTED_LANGUAGE,
    )

    translation_latency_ms = (
        time.perf_counter() - translation_start
    ) * 1000

    metrics["indic_to_english_latency_ms"] = (
        translation_latency_ms
    )

    metrics["english_query"] = english_query


    # --------------------------------------------------------
    # STEP 3 — HYBRID RAG
    # --------------------------------------------------------

    rag_start = time.perf_counter()

    retrieved_results = hybrid_search(
        english_query,
        top_k=TOP_K,
        candidate_k=CANDIDATE_K,
    )

    rag_latency_ms = (
        time.perf_counter() - rag_start
    ) * 1000

    metrics["rag_latency_ms"] = rag_latency_ms
    metrics["retrieved_chunks"] = len(
        retrieved_results
    )


    # --------------------------------------------------------
    # RAG METRICS
    # --------------------------------------------------------

    if retrieved_results:

        dense_scores = [
            float(r["dense_score"])
            for r in retrieved_results
        ]

        bm25_ranks = [
            r["bm25_rank"]
            for r in retrieved_results
            if r["bm25_rank"] is not None
        ]

        metrics["rag_top1_bge_score"] = (
            dense_scores[0]
        )

        metrics["rag_mean_bge_score"] = (
            np.mean(dense_scores)
        )

        metrics["rag_min_bge_score"] = (
            np.min(dense_scores)
        )

        metrics["rag_max_bge_score"] = (
            np.max(dense_scores)
        )

        metrics["rag_bm25_coverage_percent"] = (
            len(bm25_ranks)
            / len(retrieved_results)
            * 100
        )

        if bm25_ranks:
            metrics["rag_mean_bm25_rank"] = (
                np.mean(bm25_ranks)
            )
        else:
            metrics["rag_mean_bm25_rank"] = np.nan

        metrics["rag_top1_bm25_rank"] = (
            retrieved_results[0]["bm25_rank"]
            if retrieved_results[0]["bm25_rank"] is not None
            else np.nan
        )

    else:

        metrics["rag_top1_bge_score"] = np.nan
        metrics["rag_mean_bge_score"] = np.nan
        metrics["rag_min_bge_score"] = np.nan
        metrics["rag_max_bge_score"] = np.nan
        metrics["rag_bm25_coverage_percent"] = 0
        metrics["rag_mean_bm25_rank"] = np.nan
        metrics["rag_top1_bm25_rank"] = np.nan


    # --------------------------------------------------------
    # STEP 4 — NAVARASA / LLM
    # --------------------------------------------------------
    #
    # Temporarily replace the production Navarasa function
    # so generate_grounded_answer() behaves exactly as normal,
    # while we capture latency and token usage.
    # --------------------------------------------------------

    measured_navarasa_completion.last_metrics = {
        "latency_ms": np.nan,
        "input_tokens": np.nan,
        "output_tokens": np.nan,
        "total_tokens": np.nan,
    }

    navarasa_completion = measured_navarasa_completion

    try:

        llm_start = time.perf_counter()

        english_answer = generate_grounded_answer(
            english_query,
            retrieved_results,
        )

        llm_outer_latency_ms = (
            time.perf_counter() - llm_start
        ) * 1000

    finally:

        navarasa_completion = original_navarasa_completion

    llm_metrics = (
        measured_navarasa_completion.last_metrics
    )

    metrics["llm_latency_ms"] = (
        llm_metrics["latency_ms"]
    )

    metrics["llm_generation_wrapper_latency_ms"] = (
        llm_outer_latency_ms
    )

    metrics["llm_input_tokens"] = (
        llm_metrics["input_tokens"]
    )

    metrics["llm_output_tokens"] = (
        llm_metrics["output_tokens"]
    )

    metrics["llm_total_tokens"] = (
        llm_metrics["total_tokens"]
    )

    metrics["english_answer"] = english_answer


    # --------------------------------------------------------
    # STEP 5 — ENGLISH → INDIC
    # --------------------------------------------------------

    output_translation_start = (
        time.perf_counter()
    )

    indic_answer = translate_english_to_indic(
        english_answer,
        SELECTED_LANGUAGE,
        max_length=512,
    )

    output_translation_latency_ms = (
        time.perf_counter()
        - output_translation_start
    ) * 1000

    metrics["english_to_indic_latency_ms"] = (
        output_translation_latency_ms
    )


    # --------------------------------------------------------
    # STEP 6 — PIPER TTS
    # --------------------------------------------------------

    if SELECTED_LANGUAGE in TTS_VOICES:

        tts_start = time.perf_counter()

        paced_text = prepare_tts_text(
            indic_answer
        )

        audio_file = synthesize_tts(
            paced_text,
            SELECTED_LANGUAGE,
            TTS_OUTPUT_DIR
            / f"metrics_run_{run_number}_{SELECTED_LANGUAGE}.wav",
        )

        tts_latency_ms = (
            time.perf_counter()
            - tts_start
        ) * 1000

        metrics["tts_latency_ms"] = (
            tts_latency_ms
        )

    else:

        metrics["tts_latency_ms"] = np.nan


    # --------------------------------------------------------
    # END-TO-END
    # --------------------------------------------------------

    metrics["end_to_end_latency_ms"] = (
        time.perf_counter()
        - total_start
    ) * 1000


    # --------------------------------------------------------
    # STORE
    # --------------------------------------------------------

    records.append(metrics)

    print(
        f"ASR latency       : "
        f"{metrics['asr_latency_ms']:.2f} ms"
    )

    print(
        f"RAG latency       : "
        f"{metrics['rag_latency_ms']:.2f} ms"
    )

    print(
        f"RAG Top-1 BGE     : "
        f"{metrics['rag_top1_bge_score']:.4f}"
    )

    print(
        f"LLM latency       : "
        f"{metrics['llm_latency_ms']:.2f} ms"
    )

    if not pd.isna(metrics["llm_total_tokens"]):
        print(
            f"LLM tokens        : "
            f"{int(metrics['llm_total_tokens'])}"
        )
    else:
        print(
            "LLM tokens        : "
            "Not reported by llama-server"
        )

    if not pd.isna(metrics["tts_latency_ms"]):
        print(
            f"TTS latency       : "
            f"{metrics['tts_latency_ms']:.2f} ms"
        )

    print(
        f"End-to-end        : "
        f"{metrics['end_to_end_latency_ms']:.2f} ms"
    )


# ============================================================
# DATAFRAME
# ============================================================

metrics_df = pd.DataFrame(records)


# ============================================================
# SAVE RAW METRICS
# ============================================================

metrics_df.to_csv(
    METRICS_FILE,
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

numeric_columns = [
    "asr_latency_ms",
    "wer",
    "cer",
    "asr_accuracy_percent",
    "indic_to_english_latency_ms",
    "rag_latency_ms",
    "rag_top1_bge_score",
    "rag_mean_bge_score",
    "rag_min_bge_score",
    "rag_max_bge_score",
    "rag_bm25_coverage_percent",
    "rag_mean_bm25_rank",
    "llm_latency_ms",
    "llm_generation_wrapper_latency_ms",
    "llm_input_tokens",
    "llm_output_tokens",
    "llm_total_tokens",
    "english_to_indic_latency_ms",
    "tts_latency_ms",
    "end_to_end_latency_ms",
]

existing_numeric_columns = [
    c for c in numeric_columns
    if c in metrics_df.columns
]


summary_rows = []

for column in existing_numeric_columns:

    values = pd.to_numeric(
        metrics_df[column],
        errors="coerce",
    ).dropna()

    if len(values) == 0:
        continue

    summary_rows.append({
        "metric": column,
        "mean": values.mean(),
        "median": values.median(),
        "p95": np.percentile(values, 95),
        "min": values.min(),
        "max": values.max(),
    })


summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# DISPLAY — PER RUN
# ============================================================

print("\n")
print("=" * 75)
print("PER-RUN METRICS")
print("=" * 75)

display(
    metrics_df[
        [
            "run",
            "asr_accuracy_percent",
            "wer",
            "cer",
            "asr_latency_ms",
            "rag_latency_ms",
            "rag_top1_bge_score",
            "rag_bm25_coverage_percent",
            "llm_latency_ms",
            "llm_total_tokens",
            "english_to_indic_latency_ms",
            "tts_latency_ms",
            "end_to_end_latency_ms",
        ]
    ]
)


# ============================================================
# DISPLAY — SUMMARY
# ============================================================

print("\n")
print("=" * 75)
print("SUMMARY STATISTICS")
print("=" * 75)

display(summary_df)


# ============================================================
# PRESENTATION-FRIENDLY VALUES
# ============================================================

print("\n")
print("=" * 75)
print("DASHBOARD VALUES")
print("=" * 75)


def dashboard_value(column, suffix=""):

    values = pd.to_numeric(
        metrics_df[column],
        errors="coerce",
    ).dropna()

    if len(values) == 0:
        return "N/A"

    return f"{values.mean():.2f}{suffix}"


print(
    f"ASR Accuracy        : "
    f"{dashboard_value('asr_accuracy_percent', '%')}"
)

print(
    f"WER                 : "
    f"{dashboard_value('wer')}"
)

print(
    f"CER                 : "
    f"{dashboard_value('cer')}"
)

print(
    f"ASR Latency         : "
    f"{dashboard_value('asr_latency_ms', ' ms')}"
)

print(
    f"RAG Latency         : "
    f"{dashboard_value('rag_latency_ms', ' ms')}"
)

print(
    f"Top-1 BGE Score     : "
    f"{dashboard_value('rag_top1_bge_score')}"
)

print(
    f"BM25 Coverage       : "
    f"{dashboard_value('rag_bm25_coverage_percent', '%')}"
)

print(
    f"LLM Latency         : "
    f"{dashboard_value('llm_latency_ms', ' ms')}"
)

print(
    f"LLM Tokens/Request  : "
    f"{dashboard_value('llm_total_tokens')}"
)

print(
    f"TTS Latency         : "
    f"{dashboard_value('tts_latency_ms', ' ms')}"
)

print(
    f"End-to-End Latency  : "
    f"{dashboard_value('end_to_end_latency_ms', ' ms')}"
)


# ============================================================
# FILE LOCATION
# ============================================================

print("\n")
print("=" * 75)
print("METRICS FILE")
print("=" * 75)

print(METRICS_FILE)
print("\n✓ Evaluation complete.")
print("✓ Raw per-run data saved for chart generation.")